# Complete comparison table — EEGFormer vs. baselines

Standalone notebook. Fills in the full table:
- **EEGConformer** column (loaded from per-seed JSON + trial predictions)
- **Specificity** and **AUC** rows (reconstructed or computed)
- Wilcoxon significance shading (gray) on all metrics (p < 5%, EEGFormer > baseline)
- Saves LaTeX to `results/evaluation/wilcoxon_varying_seed_table.tex`

In [6]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score
from IPython.display import display, Markdown

In [7]:
HYP = Path('../results/hypothesis tests')

COMMON_SEEDS = {7, 42, 99, 123, 314, 456, 789, 2024, 2718, 5000}

DISPLAY_MODELS  = ['EEGNet', 'ShallowConvNet', 'CNN-LSTM', 'MultiStream',
                   'EEGConformer', 'IMCBGT', 'TGARNet', 'EEGFormer']
DISPLAY_HEADERS = ['EEGNet', 'ShallowConvNet', 'CNN-LSTM', 'Multi-Stream',
                   'EEGConformer', 'IM-CBGT', 'T-GARNet', 'EEG-TACT']

# CNN-LSTM and IMCBGT store recall/precision directly → fold-level mean±std (50 values)
# All other models → seed-level mean±std (10 per-seed means then mean/std of those)
FOLD_LEVEL = {'CNN-LSTM', 'IMCBGT'}

REF     = 'EEGFormer'
METRICS = ['accuracy', 'recall', 'precision', 'specificity', 'auc']
METRIC_LABELS = {
    'accuracy':    r'Accuracy (\%)',
    'recall':      r'Recall (\%)',
    'precision':   r'Precision (\%)',
    'specificity': r'Specificity (\%)',
    'auc':         r'AUC (\%)',
}

In [8]:
# -----------------------------------------------------------------------
# Infer positive-class prevalence per fold from CNN-LSTM.
# CNN-LSTM is the only model with explicit recall in all_fold_results.csv.
# From balanced_acc = (recall + specificity) / 2  →  specificity = 2*bacc - recall
# From accuracy = p*recall + (1-p)*specificity     →  p = (acc - spec) / (recall - spec)
# -----------------------------------------------------------------------
_cnn = pd.read_csv(HYP / 'cnn_lstm_eegnet_10seeds' / 'all_fold_results.csv')
fold_prev = {}
for fold_id, g in _cnn.groupby('fold'):
    a  = g['accuracy'].astype(float).to_numpy()
    b  = g['balanced_acc'].astype(float).to_numpy()
    r  = g['recall'].astype(float).to_numpy()
    sp = 2.0 * b - r
    dn = r - sp
    ok = np.abs(dn) > 1e-12
    fold_prev[int(fold_id)] = float(np.nanmean((a[ok] - sp[ok]) / dn[ok]))

print('Inferred positive-class prevalence per fold:')
for k in sorted(fold_prev):
    print(f'  fold {k}: {fold_prev[k]:.6f}')


def reconstruct(acc, bal, f1_val, fold_id):
    """Reconstruct recall, precision, specificity from acc/balanced_acc/F1."""
    p  = fold_prev[int(fold_id)]
    dn = 2.0 * p - 1.0
    if abs(dn) < 1e-12:
        return np.nan, np.nan, np.nan
    rec = float(np.clip((acc - 2.0 * (1.0 - p) * bal) / dn, 0.0, 1.0))
    dn2 = 2.0 * rec - f1_val
    pre = float(np.clip((f1_val * rec) / dn2, 0.0, 1.0)) if abs(dn2) > 1e-12 else np.nan
    spc = float(np.clip(2.0 * bal - rec, 0.0, 1.0))
    return rec, pre, spc

Inferred positive-class prevalence per fold:
  fold 0: 0.458333
  fold 1: 0.541667
  fold 2: 0.458333
  fold 3: 0.625000
  fold 4: 0.458333


In [9]:
# -----------------------------------------------------------------------
# Load fold-level metrics for the 7 non-EEGConformer models.
# Result: all_fold[model] has columns [seed, fold, accuracy, recall,
#         precision, specificity, auc] with 50 rows (10 seeds × 5 folds).
# -----------------------------------------------------------------------
MODEL_SOURCES = {
    'EEGNet':        ('eegnet_10seeds',         False),   # reconstruct recall/precision
    'ShallowConvNet':('shallowconvnet_10seeds',  False),
    'CNN-LSTM':      ('cnn_lstm_eegnet_10seeds', True),   # direct recall/precision
    'MultiStream':   ('multistream_10seeds',     False),
    'IMCBGT':        ('imcbgt_10seeds',          True),   # direct recall/precision
    'TGARNet':       ('tgarnet_10seeds',         False),
    'EEGFormer':     ('eegformer_10seeds',       False),
}

all_fold = {}

for mname, (folder, has_direct_rp) in MODEL_SOURCES.items():
    df = pd.read_csv(HYP / folder / 'all_fold_results.csv')
    df = df[df['seed'].astype(int).isin(COMMON_SEEDS)].copy()
    df['seed'] = df['seed'].astype(int)
    df['fold'] = df['fold'].astype(int)
    rows = []
    for _, row in df.iterrows():
        a   = float(row['accuracy'])
        b   = float(row['balanced_acc'])
        f1v = float(row['f1'])
        au  = float(row['auc'])
        if has_direct_rp:
            rec = float(row['recall'])
            pre = float(row['precision'])
            spc = float(np.clip(2.0 * b - rec, 0.0, 1.0))
        else:
            rec, pre, spc = reconstruct(a, b, f1v, int(row['fold']))
        rows.append({'seed': int(row['seed']), 'fold': int(row['fold']),
                     'accuracy': a, 'recall': rec, 'precision': pre,
                     'specificity': spc, 'auc': au})
    all_fold[mname] = pd.DataFrame(rows)

print('Loaded non-EEGConformer models:')
for m, df in all_fold.items():
    print(f'  {m}: {len(df)} rows | seeds={sorted(df.seed.unique())} | folds={sorted(df.fold.unique())}')

Loaded non-EEGConformer models:
  EEGNet: 50 rows | seeds=[np.int64(7), np.int64(42), np.int64(99), np.int64(123), np.int64(314), np.int64(456), np.int64(789), np.int64(2024), np.int64(2718), np.int64(5000)] | folds=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  ShallowConvNet: 50 rows | seeds=[np.int64(7), np.int64(42), np.int64(99), np.int64(123), np.int64(314), np.int64(456), np.int64(789), np.int64(2024), np.int64(2718), np.int64(5000)] | folds=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  CNN-LSTM: 50 rows | seeds=[np.int64(7), np.int64(42), np.int64(99), np.int64(123), np.int64(314), np.int64(456), np.int64(789), np.int64(2024), np.int64(2718), np.int64(5000)] | folds=[np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
  MultiStream: 50 rows | seeds=[np.int64(7), np.int64(42), np.int64(99), np.int64(123), np.int64(314), np.int64(456), np.int64(789), np.int64(2024), np.int64(2718), np.int64(5000)] | folds=[np.int64(0), np.in

In [10]:
# -----------------------------------------------------------------------
# EEGConformer: different data layout.
#   accuracy/bacc/f1  ← seed_*/fold_results.json
#   AUC               ← seed_*/trial_predictions.csv  (subject_mean_prob)
#   recall/precision/specificity are reconstructed like all other models
# -----------------------------------------------------------------------
conf_base = HYP / 'eegconformer_10seeds'

# Step A: compute per-(seed, fold) AUC from trial predictions
auc_conf = {}
for sd in sorted(conf_base.glob('seed_*')):
    tc = sd / 'trial_predictions.csv'
    if not tc.exists():
        continue
    t  = pd.read_csv(tc)
    sv = int(t['seed'].iloc[0])
    if sv not in COMMON_SEEDS:
        continue
    subj = t[['fold', 'subject', 'true_label', 'subject_mean_prob']].drop_duplicates()
    for fv, fg in subj.groupby('fold'):
        yt = fg['true_label'].astype(int).to_numpy()
        yp = fg['subject_mean_prob'].astype(float).to_numpy()
        auc_conf[(sv, int(fv))] = (
            float(roc_auc_score(yt, yp)) if np.unique(yt).size >= 2 else np.nan
        )

# Step B: load fold_results.json and reconstruct recall/precision/specificity
conf_rows = []
for sd in sorted(conf_base.glob('seed_*')):
    jp = sd / 'fold_results.json'
    if not jp.exists():
        continue
    with open(jp) as fh:
        entries = json.load(fh)
    for e in entries:
        sv = int(e['seed'])
        if sv not in COMMON_SEEDS:
            continue
        fv  = int(e['fold'])
        a   = float(e['test_subject_acc'])
        b   = float(e['test_subject_bacc'])
        f1v = float(e['test_subject_f1'])
        rec, pre, spc = reconstruct(a, b, f1v, fv)
        conf_rows.append({
            'seed': sv, 'fold': fv,
            'accuracy': a, 'recall': rec, 'precision': pre,
            'specificity': spc, 'auc': auc_conf.get((sv, fv), np.nan)
        })

all_fold['EEGConformer'] = pd.DataFrame(conf_rows)

print('All models loaded (expect 50 rows each):')
for m in DISPLAY_MODELS:
    df = all_fold[m]
    print(f'  {m}: {len(df)} rows, {df.seed.nunique()} seeds, {df.fold.nunique()} folds')

KeyError: 'seed'

In [ ]:
# -----------------------------------------------------------------------
# Wilcoxon signed-rank test: one-sided (EEGFormer > baseline)
# Paired on (seed, fold) → 50 pairs per test.
# Applied independently to each of the 5 metrics.
# -----------------------------------------------------------------------
ref_df = (
    all_fold[REF][['seed', 'fold'] + METRICS]
    .rename(columns={m: f'{m}_ref' for m in METRICS})
)

wilcoxon_sig  = {}  # (metric, baseline) -> bool
wilcoxon_pval = {}  # (metric, baseline) -> float

for baseline in [m for m in DISPLAY_MODELS if m != REF]:
    cmp_df = (
        all_fold[baseline][['seed', 'fold'] + METRICS]
        .rename(columns={m: f'{m}_cmp' for m in METRICS})
    )
    merged = ref_df.merge(cmp_df, on=['seed', 'fold'], how='inner')
    for metric in METRICS:
        pair = merged[[f'{metric}_ref', f'{metric}_cmp']].dropna()
        if len(pair) < 2:
            wilcoxon_sig[(metric, baseline)]  = False
            wilcoxon_pval[(metric, baseline)] = np.nan
            continue
        diff = pair[f'{metric}_ref'].to_numpy() - pair[f'{metric}_cmp'].to_numpy()
        if np.allclose(diff, 0.0):
            wilcoxon_sig[(metric, baseline)]  = False
            wilcoxon_pval[(metric, baseline)] = 1.0
        else:
            w = wilcoxon(pair[f'{metric}_ref'], pair[f'{metric}_cmp'],
                         alternative='greater', zero_method='wilcox', mode='auto')
            p = float(w.pvalue)
            wilcoxon_sig[(metric, baseline)]  = p < 0.05
            wilcoxon_pval[(metric, baseline)] = p

baselines = [m for m in DISPLAY_MODELS if m != REF]
pval_df = pd.DataFrame(
    {b: {m: round(wilcoxon_pval.get((m, b), np.nan), 4) for m in METRICS}
     for b in baselines}
).T
pval_df.index.name = 'baseline'
print('Wilcoxon p-values (one-sided: EEGFormer > baseline):')
display(pval_df)

sig_df = pd.DataFrame(
    {b: {m: wilcoxon_sig.get((m, b), False) for m in METRICS} for b in baselines}
).T
sig_df.index.name = 'baseline'
print('\nSignificant (p < 0.05):')
display(sig_df)

In [ ]:
# -----------------------------------------------------------------------
# Display aggregation:
#   CNN-LSTM, IMCBGT  → fold-level mean±std (all 50 values)
#   all other models  → seed-level mean±std (mean per seed, then mean/std of 10)
# Average accuracy rank computed across all 8 models on 50 paired folds.
# -----------------------------------------------------------------------
disp_stats = {}  # (model, metric) -> (mean_pct, std_pct)
for mname in DISPLAY_MODELS:
    df = all_fold[mname]
    for metric in METRICS:
        vals = df[metric].dropna().astype(float)
        if mname in FOLD_LEVEL:
            mean_pct = 100.0 * vals.mean()
            std_pct  = 100.0 * vals.std(ddof=1)
        else:
            seed_means = df.groupby('seed')[metric].mean().dropna()
            mean_pct = 100.0 * seed_means.mean()
            std_pct  = 100.0 * seed_means.std(ddof=1)
        disp_stats[(mname, metric)] = (float(mean_pct), float(std_pct))

# Summary table
summary_rows = []
for mname in DISPLAY_MODELS:
    row = {'model': mname}
    for metric in METRICS:
        mu, sd = disp_stats[(mname, metric)]
        row[metric] = f'{mu:.1f} \u00b1 {sd:.1f}'
    summary_rows.append(row)
print('Display statistics (mean \u00b1 std %):')
display(pd.DataFrame(summary_rows).set_index('model'))

# Average accuracy rank (all 8 models, 50 fold rows)
rank_df  = pd.DataFrame(
    {m: all_fold[m].set_index(['seed', 'fold'])['accuracy'] for m in DISPLAY_MODELS}
).dropna(axis=0, how='any')
avg_rank = rank_df.rank(axis=1, ascending=False, method='average').mean(axis=0)
print(f'\nRanked over {len(rank_df)} complete (seed, fold) pairs.')
print('Average accuracy rank (lower = better):')
display(avg_rank.sort_values().to_frame('avg_rank').round(2))

In [ ]:
# -----------------------------------------------------------------------
# Build LaTeX table, save to .tex, and display in notebook.
# Gray cell = Wilcoxon significant (p < 0.05, EEGFormer > baseline).
# Bold = best value in the row.
# -----------------------------------------------------------------------

def fmt_cell(mname, metric):
    mu, sd = disp_stats[(mname, metric)]
    if np.isnan(mu):
        return ''
    mu_r = round(mu, 1)
    sd_r = round(sd, 1) if not np.isnan(sd) else 0.0
    row_best = max(
        round(disp_stats[(m, metric)][0], 1)
        for m in DISPLAY_MODELS
        if not np.isnan(disp_stats[(m, metric)][0])
    )
    inner = f'{mu_r:.1f} \\pm {sd_r:.1f}'
    if mu_r == row_best:
        inner = f'\\mathbf{{{inner}}}'
    body = f'${inner}$'
    if mname != REF and wilcoxon_sig.get((metric, mname), False):
        body = f'\\cellcolor{{lightgray}}{body}'
    return body


SEP = '\n& '
END = r'\\'

lines = [
    r'\begin{tabular}{lcccccccc}',
    r'\toprule',
    (r'\textbf{Model}' + SEP
     + SEP.join(f'\\textbf{{{h}}}' for h in DISPLAY_HEADERS) + END),
    r'\midrule',
    r'\multicolumn{9}{c}{\textbf{Varying seed (average over 50 folds)}} \\',
    r'\midrule',
]

for metric in METRICS:
    cells = SEP.join(fmt_cell(m, metric) for m in DISPLAY_MODELS)
    lines.append(METRIC_LABELS[metric] + SEP + cells + END)

# Average rank row
rank_vals = [avg_rank.get(m, np.nan) for m in DISPLAY_MODELS]
best_rank = min(v for v in rank_vals if not np.isnan(v))
rank_cells = [
    (f'$\\mathbf{{{rv:.2f}}}$' if abs(rv - best_rank) < 1e-9 else f'${rv:.2f}$')
    if not np.isnan(rv) else ''
    for rv in rank_vals
]
lines.append('Average Rank' + SEP + SEP.join(rank_cells) + END)
lines += [r'\bottomrule', r'\end{tabular}']

latex_out = '\n'.join(lines)

# Save
out_tex = Path('../results/evaluation/wilcoxon_varying_seed_table.tex')
out_tex.parent.mkdir(parents=True, exist_ok=True)
out_tex.write_text(latex_out, encoding='utf-8')
print(f'Saved to: {out_tex.resolve()}\n')

# Display in notebook
display(Markdown(f'### Generated LaTeX\n\n```latex\n{latex_out}\n```'))